# Build the Root Practice Lab

**Learning objective:** Evaluate every (feature, threshold) candidate on a tiny dataset, track impurity reduction, and choose the **root** split with the lowest weighted Gini.

Features: `age`, `resting_bp`. Target: `heart_disease`.


In [ ]:
import numpy as np
import pandas as pd

df = pd.DataFrame([
    {"age": 42, "resting_bp": 118, "heart_disease": 0},
    {"age": 45, "resting_bp": 122, "heart_disease": 0},
    {"age": 50, "resting_bp": 130, "heart_disease": 0},
    {"age": 53, "resting_bp": 138, "heart_disease": 1},
    {"age": 55, "resting_bp": 142, "heart_disease": 1},
    {"age": 60, "resting_bp": 150, "heart_disease": 1},
])
print(df)


In [ ]:
def gini_impurity(y):
    y = np.asarray(y)
    if len(y) == 0:
        return 0.0
    counts = np.bincount(y)
    props = counts / counts.sum()
    return float(1.0 - np.sum(props ** 2))

def weighted_gini(y_left, y_right):
    n_l, n_r = len(y_left), len(y_right)
    n = n_l + n_r
    return 0.0 if n == 0 else (n_l / n) * gini_impurity(y_left) + (n_r / n) * gini_impurity(y_right)

def candidate_thresholds(values):
    uniq = sorted(set(values))
    return [(uniq[i] + uniq[i + 1]) / 2 for i in range(len(uniq) - 1)]

parent_gini = gini_impurity(df["heart_disease"].to_numpy())
print(f"Parent Gini: {parent_gini:.3f}")
print("Age thresholds:", candidate_thresholds(df["age"]))
print("BP thresholds:", candidate_thresholds(df["resting_bp"]))


## Guided example — score every candidate

Loop over features and thresholds, compute weighted Gini and **impurity reduction** = parent Gini − weighted Gini.


In [ ]:
features = ["age", "resting_bp"]
records = []
y = df["heart_disease"].to_numpy()

for feature in features:
    for t in candidate_thresholds(df[feature]):
        left = df[df[feature] <= t]["heart_disease"].to_numpy()
        right = df[df[feature] > t]["heart_disease"].to_numpy()
        wg = weighted_gini(left, right)
        records.append({
            "feature": feature,
            "threshold": t,
            "weighted_gini": round(wg, 4),
            "reduction": round(parent_gini - wg, 4),
            "n_left": len(left),
            "n_right": len(right),
        })

candidates = pd.DataFrame(records).sort_values("weighted_gini")
print(candidates)


## Exercise 1 — select the root

From `candidates`, select the row with the lowest `weighted_gini`. Store it as `root_split`.

**Expected:** a feature/threshold pair with the largest impurity reduction (tied with lowest weighted Gini).


In [ ]:
# TODO Exercise 1
root_split = candidates.iloc[0]
print("Root split:")
print(root_split)


## Exercise 2 — verify the partition

Using `root_split`, recreate left/right groups and print each group's labels and Gini.


In [ ]:
# TODO Exercise 2
feat = root_split["feature"]
thr = root_split["threshold"]
left_df = df[df[feat] <= thr]
right_df = df[df[feat] > thr]
print("Left labels:", left_df["heart_disease"].tolist(), "Gini:", round(gini_impurity(left_df["heart_disease"]), 3))
print("Right labels:", right_df["heart_disease"].tolist(), "Gini:", round(gini_impurity(right_df["heart_disease"]), 3))


## Reflection

1. Why is the best split the minimum weighted Gini, not the maximum?
2. Could two different (feature, threshold) pairs produce the same reduction?
3. What should happen next if a child group is still impure?
